## Creating vp_data4

The first step in creating database *vp_data4.db* that should be analogous to *vp_data3.db*. The difference is that both one-member and two-member patterns are included in *patterns* table and their occurrence in transactions is not checked. However, still only patterns that contain up to one compound part and do not contain a word belonging into *other* category are included. 

Currently, only table *patterns* is created, based on which other tables can be created in future. The table will be exported to *target_data* directory in CSV-format.

In [28]:
import sys

sys.path.append('../../../common_code')

In [29]:
import sqlite3
from db_operations.db_display import *

## Input parameters

In [30]:
INPUT_DIR = "C:/Users/liivas/Documents/Töö/verbirektisoonid"
DB_DIR = "../001_creating_pattern_tables"

RESULT_DB = "vp_data4.db"
TRANSACTION_DB = f"{INPUT_DIR}/v32_data.db"
VERB_PATTERNS_DB = f"{DB_DIR}/verb_patterns_new.db"
TEMP_LEN2_DB = "vp_len2.db"

## Data processing

In [19]:
con = sqlite3.connect(RESULT_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS v32')
cur.execute(f'ATTACH DATABASE "{TEMP_LEN2_DB}" AS len2')
cur.execute(f'ATTACH DATABASE "{VERB_PATTERNS_DB}" AS vp')

### I table patterns

Veerud:

    pat_id - (mustri ID tabelis patterns_len1)
    pattern - (algne muster sõnena)
    verb_word - (mustri (pea)verb)
    verb_compound - (pikema verbiühendi ülejäänud osad)
    phrase_nr - (fraasi number (1 või 2)
    phrase_case - (fraasi põhiliikme (pärast verbi) kääne)
    adp - (kaassõna)
    inf_verb - (infiniitverb)
    
Vajalik info üheliikmeliste mustrite kohta saadakse tabelitest patterns_len1 (*verb_patterns_new.db*). Kaheliikmeliste mustrite info saadakse tabelist temp_patterns (*vp_len2.db*).

In [25]:
cur.execute("""
DROP TABLE IF EXISTS patterns
""")

cur.execute("""
CREATE TABLE patterns AS
SELECT
    ID AS pat_id,
    word || ' ' || government AS pattern,
    verb_word,
    compound_prt1 AS verb_compound,
    phrase_nr,
    pat.w_case AS phrase_case,
    adp,
    pat.verb AS inf_verb
FROM 
    vp.patterns_len1 as pat
WHERE 
    compound_prt2 = ''
AND 
    compound_prt3 = ''
AND 
    other = ''
""")

cur.execute("""
INSERT INTO patterns
SELECT *
FROM
    len2.temp_patterns
""")
    
cur.execute("""
CREATE INDEX pat_id_idx ON patterns(pat_id)
"""
)

cur.execute("""
CREATE INDEX phrase_case_idx ON patterns(phrase_case)
"""
)

cur.execute("""
CREATE INDEX adp_idx ON patterns(adp)
"""
)

cur.execute("""
CREATE INDEX inf_verb_idx ON patterns(inf_verb)
"""
)

cur.execute("""
CREATE INDEX verb_word_idx ON patterns(verb_word)
"""
)

cur.execute("""
CREATE INDEX verb_compound_idx ON patterns(verb_compound)
"""
)

cur.execute("""
CREATE INDEX phrase_nr_idx ON patterns(phrase_nr)
"""
)

con.commit()
con.close()

## Result

In [31]:
display_sqlite_as_dataframe(RESULT_DB, 'patterns', 10)

,pat_id,pattern,verb_word,verb_compound,phrase_nr,phrase_case,adp,inf_verb
0,1,aasima keda*,aasima,,1,part,,
1,2,aasima kelle kallal,aasima,,1,gen,kallal,
2,3,abielluma kellega,abielluma,,1,kom,,
3,4,abikätt ulatama kellele,ulatama,abikätt,1,all,,
4,5,abstraheeruma millest/kellest,abstraheeruma,,1,el,,
5,6,adresseerima mida*,adresseerima,,1,part,,
6,7,adresseerima kellele,adresseerima,,1,all,,
7,8,aevastama mille peale,aevastama,,1,gen,peale,
8,9,agiteerima keda*,agiteerima,,1,part,,
9,10,ahistama keda*,ahistama,,1,part,,
